## MOdel Train,Evaluate,Pridiction

In [12]:
import pandas as pd
import logging
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Configure logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler("analysis.log"), logging.StreamHandler()])

# Load the datasets
try:
    train_data = pd.read_csv('../data/train.csv', low_memory=False)
    logging.info("Loaded training data successfully.")
except FileNotFoundError:
    logging.error("Training data file not found.")

try:
    test_data = pd.read_csv('../data/test.csv', low_memory=False)
    logging.info("Loaded test data successfully.")
except FileNotFoundError:
    logging.error("Test data file not found.")

# Check column names
logging.info("Train Data Columns: %s", train_data.columns.tolist())
logging.info("Test Data Columns: %s", test_data.columns.tolist())

# Convert 'Date' to datetime if the column exists
if 'Date' in train_data.columns:
    train_data['Date'] = pd.to_datetime(train_data['Date'])
if 'Date' in test_data.columns:
    test_data['Date'] = pd.to_datetime(test_data['Date'])

# Convert datetime columns to numerical features
if 'Date' in train_data.columns:
    train_data['Year'] = train_data['Date'].dt.year
    train_data['Month'] = train_data['Date'].dt.month
    train_data['Day'] = train_data['Date'].dt.day

if 'Date' in test_data.columns:
    test_data['Year'] = test_data['Date'].dt.year
    test_data['Month'] = test_data['Date'].dt.month
    test_data['Day'] = test_data['Date'].dt.day

# Drop the original 'Date' column if it's no longer needed
train_data.drop('Date', axis=1, inplace=True, errors='ignore')
test_data.drop('Date', axis=1, inplace=True, errors='ignore')

# Identify categorical columns
categorical_cols = train_data.select_dtypes(include=['object']).columns.tolist()

# One-hot encode categorical variables
train_data = pd.get_dummies(train_data, columns=categorical_cols, drop_first=True)
test_data = pd.get_dummies(test_data, columns=categorical_cols, drop_first=True)

# Align train and test data (in case of different categories)
train_data, test_data = train_data.align(test_data, join='left', axis=1, fill_value=0)

# Ensure target column is not present in test_data
if 'Sales' in test_data.columns:
    test_data.drop('Sales', axis=1, inplace=True)

# Step 1: Define features and target variable
X = train_data.drop('Sales', axis=1)  # Features
y = train_data['Sales']              # Target variable

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Train the model
model = RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
logging.info("Model training completed.")

# Step 4: Make predictions on the validation set
val_predictions = model.predict(X_val)

# Step 5: Evaluate the model
mae = mean_absolute_error(y_val, val_predictions)
logging.info(f'Mean Absolute Error: {mae}')

# Step 6: Align test_data columns to match training features
logging.info("Aligning test_data columns with training features...")
test_data = test_data[X_train.columns]  # Align test data with training features
logging.debug(f"Training features: {X_train.columns.tolist()}")
logging.debug(f"Test features: {test_data.columns.tolist()}")

# Make predictions on the test data
test_predictions = model.predict(test_data)
logging.info("Predictions on test data completed.")

2025-01-04 17:00:56,753 - INFO - Loaded training data successfully.
2025-01-04 17:00:56,920 - INFO - Loaded test data successfully.
2025-01-04 17:00:56,927 - INFO - Train Data Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
2025-01-04 17:00:56,933 - INFO - Test Data Columns: ['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
2025-01-04 17:01:27,158 - INFO - Model training completed.
2025-01-04 17:01:27,863 - INFO - Mean Absolute Error: 504.2116047817068
2025-01-04 17:01:27,865 - INFO - Aligning test_data columns with training features...
2025-01-04 17:01:27,922 - INFO - Predictions on test data completed.


### save the model

In [13]:
import joblib
joblib.dump(model, 'sales_forecasting_model.pkl')

['sales_forecasting_model.pkl']

### Analyze the features importance

In [16]:
import numpy as np
features_importance = model.feature_importances_
indices = np.argsort(features_importance)[::-1]
logging.info("Feature Importance:")
for f in range(X_train.shape[1]):
    feature_name = X_train.columns[indices[f]]  # Ensure you're using the correct DataFrame (x or X_train)
    importance_score = features_importance[indices[f]]
    logging.info(f"{f + 1}. Feature: {feature_name} - Importance: {importance_score:.4f}")

2025-01-04 17:15:11,213 - INFO - Feature Importance:
2025-01-04 17:15:11,217 - INFO - 1. Feature: Customers - Importance: 0.8594
2025-01-04 17:15:11,219 - INFO - 2. Feature: Store - Importance: 0.0835
2025-01-04 17:15:11,221 - INFO - 3. Feature: Promo - Importance: 0.0238
2025-01-04 17:15:11,222 - INFO - 4. Feature: DayOfWeek - Importance: 0.0102
2025-01-04 17:15:11,225 - INFO - 5. Feature: Day - Importance: 0.0095
2025-01-04 17:15:11,226 - INFO - 6. Feature: Month - Importance: 0.0091
2025-01-04 17:15:11,229 - INFO - 7. Feature: Year - Importance: 0.0029
2025-01-04 17:15:11,231 - INFO - 8. Feature: SchoolHoliday - Importance: 0.0015
2025-01-04 17:15:11,233 - INFO - 9. Feature: StateHoliday_a - Importance: 0.0001
2025-01-04 17:15:11,235 - INFO - 10. Feature: StateHoliday_b - Importance: 0.0000
2025-01-04 17:15:11,238 - INFO - 11. Feature: StateHoliday_c - Importance: 0.0000
2025-01-04 17:15:11,241 - INFO - 12. Feature: Open - Importance: 0.0000
